# Optimal control of a single spin by using GOAT over GRAPE

In this introductory example we compute the gradients of analytic pulse shapes in [GOAT](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.120.150401) by using the gradients of the time evolution from [GRAPE](https://www.sciencedirect.com/science/article/pii/S1090780704003696).
This is performed by using chain rule - 
$$\frac{\partial J}{\partial p} = \sum_k \frac{\partial J}{\partial c_k} \frac{\partial c_k}{\partial p}$$
where $c_k = c(t_k)$ the 'pixelated' control pulse, $\vec{p}$ are the analytical parameters of the control pulse $c(t) \equiv c(\vec{p}, t)$, and $\frac{\partial J}{\partial c_k}$ are the gradients from GRAPE.

## 1. Generate a PWC pulse shape

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from paraqeet.model.drive import HermitianDriveGenerator
from paraqeet.model.qubit import Qubit
from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import FlatTopGaussianEnvelope
from paraqeet.signal.pwc_generator import PWCGenerator
from paraqeet.signal.waveform import FlatTopGaussianFilter

Define the `FlatTopGaussianEnvelope` and the `PWCGenerator`. The `PWCGenerator` is used to produce the pixelated pulse shape for computing the gradients using GRAPE. 

Here we also define the `FlatTopGaussianFilter` to ensure that the pulse always starts and end at zero.

The optimization would be performed on the `FlatTopGaussianEnvelope` parameters: `amplitude`, `t_up`, `t_down`, `ramp_time`.

In [ ]:
t_final = 20e-9
num_pwc = 30  # number of signal pixels
t_simu = t_final
times = np.linspace(0, t_final, num_pwc)

tone = FlatTopGaussianEnvelope(
    amplitude=Quantity(np.pi / t_final / 3, -np.pi / t_final, np.pi / t_final, name="Amplitude"),
    t_up=Quantity(1e-9, 0.0, t_final, name="t_up"),
    t_down=Quantity(t_final - 1e-9, 0.0, t_final, name="t_down"),
    ramp_time=Quantity(1e-9, 0.5e-9, t_final, name="ramp_time"),
)

filtered_tone = FlatTopGaussianFilter(tone, t_final=Quantity(t_final, 0.0, 1.2 * t_final, name="t_final"))

gen = PWCGenerator(envelopes=[filtered_tone], tlist=times)

In [ ]:
params = gen.get_parameters()
params[0].set_limits(-200e6, 200e6)
params[1].set_limits(-200e6, 200e6)

In [ ]:
from plotting import plot_signal

ts = np.linspace(0, t_final, 501)
fig, ax = plt.subplots(1, figsize=(5, 3))

plot_signal(filtered_tone, ts, ax, linestyle="-", label="Smooth")
plot_signal(gen, ts, ax, linestyle="--", label="PWC")

## 2. Define Hamiltonian in the rotating frame of drive

As a simple toy model, we use a single spin.

In [ ]:
from paraqeet.model.rotating_frame import RotatingFrameDrive
from paraqeet.model.schroedinger_equation import SchroedingerEquation
from paraqeet.model.system import OpenSystem
from paraqeet.quantity import Array

controlled_qubit = Qubit(frequency=Quantity(0.0, 0.0, 2 * np.pi * 1e6, unit="Hz"), drives=[])
drive = HermitianDriveGenerator(controlled_qubit.sigma_minus, gen)
controlled_qubit.drives = [drive]
# spin = SpinRWA(drives=[drive])
model = SchroedingerEquation(
    hamiltonian_func=controlled_qubit.get_hamiltonian,
    hamiltonian_and_gradient_func=controlled_qubit.get_hamiltonian_and_gradient,
)

Using GRAPE as the method to propagate and compute the gradients

In [ ]:
from paraqeet.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from paraqeet.measurement.utils import overlap_state_vector
from paraqeet.propagation.scipy_expm_grape import ScipyExpmGRAPE
from paraqeet.propagation.utils import grape_operator_sandwich_function_closed

init = np.array([[1.0], [0.0]])  # |0>
target = np.array([[0.0], [1.0]])  # |1>

prop = ScipyExpmGRAPE(
    eom_func=model.get_value,
    eom_and_grad_func=model.get_value_and_gradient,
    resolution=2e9,
    initial_state=init,
    target_state=target,
    operator_sandwich_function=grape_operator_sandwich_function_closed,
)

zeroone = StateTransferFidelityGRAPE(
    propagation_func=prop.propagate,
    propagation_and_gradient_func=prop.get_value_and_gradient,
    target_state=target,
    overlap=overlap_state_vector,
)

In [ ]:
zeroone.measure(times)

In [ ]:
from plotting import plot_signal_and_dynamics

ts = np.linspace(0.0, t_final, 101)
plot_signal_and_dynamics(gen, prop, ts, state_labels=[r"$|0\rangle$", r"$|1\rangle$"])

## 3. Optimization
Finally, we define the `GOATOverGRAPE` fideltiy that chains together the GRAPE gradients to compute the gradient wrt the tone parameters

In [ ]:
from paraqeet.measurement.goat_over_grape import GOATOverGRAPE
from paraqeet.optimization_map import OptimizationMap
from paraqeet.optimizers.scipy_optimizer_gradient import ScipyOptimizerGradient

optmap = OptimizationMap()
optmap.add(filtered_tone)
optmap.register_params_with_optimizables()

goat = GOATOverGRAPE(zeroone, gen, prop.resolution)
opt_grad = ScipyOptimizerGradient(measure_and_gradient_func=goat.get_value_and_gradient, optimization_map=optmap)

In [ ]:
opt_grad.optimize(np.array([0.0, t_simu]))

For this simple example, we reach near perfect fidelity within a few iterations.

In [ ]:
plot_signal_and_dynamics(gen, prop, ts, state_labels=[r"$|0\rangle$", r"$|1\rangle$"])